### Cell A ｜ 环境与 DashScope client

In [1]:
# Cell A: 环境与 DashScope client
import os, sys, json, random
from pathlib import Path
from openai import OpenAI
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("src")
API_KEY = os.environ.get("DASHSCOPE_API_KEY", "")
assert API_KEY, "❌ 没读到 DASHSCOPE_API_KEY，重启 VS Code / Jupyter 后重试"
client = OpenAI(api_key=API_KEY, base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
MODEL_NAME = "qwen3-32b"
print(f"✅ Client ready, model = {MODEL_NAME}")

✅ Client ready, model = qwen3-32b


### Cell B ｜ 验证

In [2]:
# Cell B: 验证 §5.1 改造
# 不是调原生 client.chat.completions.create——是调 §5.1 改完的 get_completion_1，
# 确认改造后 Piao 内部代码能正常工作。这是 §五正式跑的硬前置。
from utils import get_completion_1
resp = get_completion_1("用一句中文说你好。")
print("Raw response:", resp)
assert resp and len(resp) > 0, "❌ get_completion_1 返回空"
print("✅ §5.1 改造验证通过，可以去 §5.3 写 shell 脚本")

Raw response: 你好！
✅ §5.1 改造验证通过，可以去 §5.3 写 shell 脚本


### Cell 1

In [3]:
# Cell L1: 单 agent micro smoke（验证 simulate_debiased.User + initialize_tweet_debias 链路）
import sys; sys.path.insert(0, "src")
import importlib, utils, simulate_debiased
importlib.reload(utils); importlib.reload(simulate_debiased)
from simulate_debiased import User

var_dict = {
    "environment": "Sociopolitical", "topic": "Politics",
    "S_m2": "strongly support the Republican party",
    "S_m1": "support the Republican party",
    "S_0":  "don't have a tendency",
    "S_p1": "support the Democratic party",
    "S_p2": "Strongly support the Democratic party",
    "S_m2_e": "the Republic party is absolutely better than the Democratic party in every aspect.",
    "S_m1_e": "the Republican party and the Democratic party both have ups and downs, but the Republican party have a slight edge.",
    "S_0_e":  "doesn't lean towards or favor either the Democratic or Republican party.",
    "S_p1_e": "the Democratic party and the Republican party both have ups and downs, but the Democratic party have a slight edge.",
    "S_p2_e": "the Democratic party is absolutely better than the Republican party in every aspect.",
    "side_b_0": "Support the Democratic party",
    "side_s_0": "Support the Republican party",
    "side_e_0": "Maintain neutrality",
}

# 强制触发 init（node_id=0 在 §5.1b 改动 1 的 <16 保底范围里）
u = User(node_id=0, profile={"side": 2, "p_side": 2}, message_list=[],
         friend_pool=[1, 2, 3], var_dict=var_dict, probability=0.9)

print("✅ User 创建成功")
print("message_list 长度:", len(u.message_list))
print("init 后第一条消息:", u.message_list[0] if u.message_list else "(空)")
print("profile keys:", list(u.profile.keys()))

Expecting value: line 1 column 1 (char 0)
✅ User 创建成功
message_list 长度: 1
init 后第一条消息: {'source': 0, 'target': None, 'content': 'The Democratic Party champions equality, climate action, and healthcare for all. They stand for progress and protecting marginalized communities. With bold vision and compassion, they lead the way toward a fairer future. I strongly support the Democratic Party.'}
profile keys: ['side', 'p_side', 'reasons', 'tendency']


### Cell 2 ｜ 生成 WS_5 极小网络

In [ ]:
# Cell 2: 生成 5 节点 Watts-Strogatz 小网络
N, K, P, SEED = 5, 4, 0.1, 42
random.seed(SEED); np.random.seed(SEED)

g = nx.watts_strogatz_graph(n=N, k=K, p=P, seed=SEED)
print(f"WS 网络 N={N} k={K} p={P}")
print(f"边 {g.number_of_edges()}, 平均度 {2*g.number_of_edges()/N:.2f}, 聚类 {nx.average_clustering(g):.3f}")

dg = g.to_directed()
out_dir = Path("data/WS_5")
out_dir.mkdir(parents=True, exist_ok=True)

# edges.csv: Piao 真实格式 source,target（已对照源码核验）
pd.DataFrame([(u, v) for u, v in dg.edges()], columns=["source", "target"]).to_csv(
    out_dir / "edges.csv", index=False)

# data_ID2Net_ID.csv: Piao 真实格式单列 Network_id（simulate.py usecols=["Network_id"]）
pd.DataFrame({"Network_id": list(range(N))}).to_csv(
    out_dir / "data_ID2Net_ID.csv", index=False)

# user_message_generate.json: 每节点必须有 key（Piao 启动会读，缺 key 会 KeyError）
with open(out_dir / "user_message_generate.json", "w") as f:
    json.dump({str(i): [] for i in range(N)}, f)

print(f"✅ 三件套落地到 {out_dir}/")
fig, ax = plt.subplots(figsize=(7, 7))
nx.draw_circular(g, node_size=50, alpha=0.7, ax=ax, edge_color="gray")
ax.set_title(f"WS N={N} k={K} p={P}"); plt.show()

### Cell C ｜ 生成 WS_80 小网络

In [ ]:
# Cell C: 生成 80 节点 Watts-Strogatz 小网络
N, K, P, SEED = 80, 6, 0.1, 42
random.seed(SEED); np.random.seed(SEED)

g = nx.watts_strogatz_graph(n=N, k=K, p=P, seed=SEED)
print(f"WS 网络 N={N} k={K} p={P}")
print(f"边 {g.number_of_edges()}, 平均度 {2*g.number_of_edges()/N:.2f}, 聚类 {nx.average_clustering(g):.3f}")

dg = g.to_directed()
out_dir = Path("data/WS_80")
out_dir.mkdir(parents=True, exist_ok=True)

# edges.csv: Piao 真实格式 source,target（已对照源码核验）
pd.DataFrame([(u, v) for u, v in dg.edges()], columns=["source", "target"]).to_csv(
    out_dir / "edges.csv", index=False)

# data_ID2Net_ID.csv: Piao 真实格式单列 Network_id（simulate.py usecols=["Network_id"]）
pd.DataFrame({"Network_id": list(range(N))}).to_csv(
    out_dir / "data_ID2Net_ID.csv", index=False)

# user_message_generate.json: 每节点必须有 key（Piao 启动会读，缺 key 会 KeyError）
with open(out_dir / "user_message_generate.json", "w") as f:
    json.dump({str(i): [] for i in range(N)}, f)

print(f"✅ 三件套落地到 {out_dir}/")
fig, ax = plt.subplots(figsize=(7, 7))
nx.draw_circular(g, node_size=50, alpha=0.7, ax=ax, edge_color="gray")
ax.set_title(f"WS N={N} k={K} p={P}"); plt.show()

### Cell D 输出可视化

In [ ]:
# Cell D: 读 Piao 原版输出可视化
# 改 EPOCH 和 piao_out 路径以匹配你跑的版本（smoke=5 或正式=50）
EPOCH = 5  # 正式跑改成 50
piao_out = Path(f"output_pol/e{EPOCH}_prob0.1,0.2,0.4,0.2,0.1_data/WS_80_Politics")
assert piao_out.exists(), f"❌ 找不到 {piao_out}，先跑 run_piao.bat"

records = []
for ep in range(EPOCH + 1):  # 0 ~ EPOCH
    with open(piao_out / f"profile_{ep}.json") as f:
        profile = json.load(f)
    for node_id, p in profile.items():
        records.append({"t": ep, "agent_id": int(node_id), "side": p["side"]})
df_piao = pd.DataFrame(records)
print(f"读入 {len(df_piao)} 条，{df_piao['agent_id'].nunique()} agents × {df_piao['t'].nunique()} epochs")

# 三阵营聚合（side 已是数值化 -2/-1/0/+1/+2）
df_piao["camp"] = df_piao["side"].map(lambda s: "Rep" if s < 0 else ("Dem" if s > 0 else "Mid"))
camp_props = df_piao.groupby("t")["camp"].value_counts(normalize=True).unstack(fill_value=0)
print("\n=== 三阵营占比演化 ==="); print(camp_props)

# 三指标
rho = df_piao.assign(is_mid=(df_piao['side']==0)).groupby('t')['is_mid'].mean()
var_s = df_piao.groupby('t')['side'].var()
mean_s = df_piao.groupby('t')['side'].mean()
print(f"\nρ_mid (中间立场占比): {rho.to_dict()}")
print(f"方差 (锐化指标):     {var_s.to_dict()}")
print(f"平均立场 (方向):     {mean_s.to_dict()}")
print(f"\n方差变化 {var_s.iloc[0]:.2f} → {var_s.iloc[-1]:.2f} ({(var_s.iloc[-1]/var_s.iloc[0]-1)*100:+.1f}%)")
print(f"对比 Piao 报告: ρ_mid 应从 40% → 0.4-22.5%")

fig, ax = plt.subplots(figsize=(10, 5))
for c, color in [("Rep", "red"), ("Mid", "gray"), ("Dem", "blue")]:
    if c in camp_props.columns:
        ax.plot(camp_props.index, camp_props[c], marker="o", label=c, color=color, linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("阵营占比")
ax.set_title(f"Piao 原版 simulate_debiased.py 三阵营演化 (N=80, t={EPOCH})")
ax.legend(); ax.grid(alpha=0.3); plt.show()